# Notebook 4 — Serie temporali e turning points
**Tesi**: Stance detection sui commenti YouTube come predittore del referendum 2026

Questo notebook alimenta il **Capitolo 5** (RQ2 — turning points) e produce i grafici richiesti dal TODO §1.5: "tre linee" (referendum / governo / stampa) e "grafico solo stance".

| Sezione | Cosa fa | Riferimento tesi |
|---------|---------|------------------|
| **§1** Setup e 3 serie | Costruisce serie giornaliera deduplicata per ciascun corpus | Cap 3, §3.5.1 |
| **§2** Tre linee sovrapposte | Plot referendum + governo + stampa (raw + MA7) | TODO §1.5 |
| **§3** Solo stance | Plot serie referendum stand-alone con linea voto reale | TODO §1.5 |
| **§4** Turning points automatici | Derivata locale + soglia per identificare i punti salienti | Cap 3, §3.5.5 |
| **§5** Verifica falsificabile di H2 | Precisione/recall vs repertorio eventi | Cap 5, §5.4, H2 |
| **§6** Top video per turning point | Per ogni TP: 5 video più commentati nella finestra ±2gg | Cap 5, §5.2 |
| **§7** Articoli stampa per turning point | Per ogni TP: 5 articoli stampa nella finestra ±7gg | Cap 5, §5.2 |
| **§8** Parole chiave per TP | LDA-like: parole più frequenti commenti vs titoli stampa | Cap 5, §5.4 |
| **§9** Top picchi stampa | Top 10 picchi pro-Sì / pro-No e inversioni della serie stampa | Cap 6 (preview) |

**Input atteso da notebook 03:** `referendum_deduplicato.csv`. Se non disponibile, ricostruito al volo dal raw.

## §1 — Setup e ricostruzione delle 3 serie giornaliere

Carico i tre dataset di inferenza e costruisco per ciascuno una serie temporale giornaliera della % pro-Sì (per stampa: % pro-riforma).

**Convenzione metodologica:**
- YouTube: deduplicazione per `(Autore, Giorno)` con `keep='last'` — coerente con §3.4 della tesi.
- Stampa: nessuna deduplicazione (ogni articolo è un'unità autonoma).
- Filtro: solo giornate con almeno 5 commenti/articoli per ridurre la varianza spuria nei giorni a basso volume.
- Calcolo MA7 per la visualizzazione, ma la serie *grezza* è quella usata per l'identificazione automatica dei TP.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

PATH_REFERENDUM = "/kaggle/input/datasets/francescochiarolanza/commenti-youtube-referendum-giustizia/risultati_inferenza_completa (2).csv"
PATH_GOVERNO    = "/kaggle/input/datasets/francescochiarolanza/dataset-consenso-meloni/inferenza_definitiva.csv"
PATH_STAMPA     = "/kaggle/input/datasets/francescochiarolanza/titoli-giornali-referendum-giustizia/dataset_articoli_referendum.csv"

SI_REAL = 46.1
NO_REAL = 53.9
DATA_VOTO = pd.Timestamp("2026-03-22")


def load_youtube(path, nome, col_stance="PREDIZIONE_STANCE", col_data="Data_Commento"):
    """Serie giornaliera Sì% su aggregato deduplicato per (Autore, Giorno)."""
    try:
        df = pd.read_csv(path, dtype=str, encoding="utf-8-sig",
                         engine="python", on_bad_lines="skip")
        if len(df.columns) <= 3:
            df = pd.read_csv(path, sep=";", dtype=str, encoding="utf-8-sig",
                             engine="python", on_bad_lines="skip")
    except Exception:
        df = pd.read_csv(path, sep=";", dtype=str, encoding="utf-8-sig",
                         engine="python", on_bad_lines="skip")
    df = df.drop(columns=[c for c in df.columns if "Unnamed" in c], errors="ignore")
    df["Data"] = pd.to_datetime(df[col_data], errors="coerce", dayfirst=True, utc=True)
    df = df.dropna(subset=["Data"]).copy()
    df["Data"] = df["Data"].dt.tz_localize(None)
    df["STANCE"] = df[col_stance].astype(str).str.strip().str.upper()
    df = df[df["STANCE"].isin(["POS", "NEG"])]
    df["Giorno"] = df["Data"].dt.date

    # Deduplicazione un autore-un voto giornaliero
    if "Autore" in df.columns:
        df = df.sort_values("Data").drop_duplicates(subset=["Autore", "Giorno"], keep="last")

    daily = df.groupby(["Giorno", "STANCE"]).size().unstack(fill_value=0)
    for c in ["POS", "NEG"]:
        if c not in daily.columns:
            daily[c] = 0
    daily["TOT"] = daily["POS"] + daily["NEG"]
    daily = daily[daily["TOT"] >= 5]
    daily.index = pd.to_datetime(daily.index)
    daily["SI_PCT"] = daily["POS"] / daily["TOT"] * 100
    daily["MA7"] = daily["SI_PCT"].rolling(7, min_periods=3).mean()
    daily["DELTA"] = daily["SI_PCT"].diff()

    print(f"✅ {nome}: {len(df):,} commenti deduplicati, {len(daily)} giorni")
    return daily, df


def load_stampa(path):
    """Serie giornaliera Sì% degli articoli stampa (no deduplicazione)."""
    try:
        df = pd.read_csv(path, dtype=str, encoding="utf-8-sig",
                         engine="python", on_bad_lines="skip")
        if len(df.columns) <= 3:
            df = pd.read_csv(path, sep=";", dtype=str, encoding="utf-8-sig",
                             engine="python", on_bad_lines="skip")
    except Exception:
        df = pd.read_csv(path, sep=";", dtype=str, encoding="utf-8-sig",
                         engine="python", on_bad_lines="skip")
    df = df.drop(columns=[c for c in df.columns if "Unnamed" in c], errors="ignore")
    date_col   = next((c for c in df.columns if "data" in c.lower()), None)
    stance_col = next((c for c in df.columns if "stance" in c.lower()), None)
    df["Data"] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=["Data"]).copy()
    df["STANCE"] = df[stance_col].astype(str).str.strip().str.upper()
    df = df[df["STANCE"].isin(["POS", "NEG"])]
    df["Giorno"] = df["Data"].dt.date

    daily = df.groupby(["Giorno", "STANCE"]).size().unstack(fill_value=0)
    for c in ["POS", "NEG"]:
        if c not in daily.columns:
            daily[c] = 0
    daily["TOT"] = daily["POS"] + daily["NEG"]
    daily = daily[daily["TOT"] >= 1]
    daily.index = pd.to_datetime(daily.index)
    daily["SI_PCT"] = daily["POS"] / daily["TOT"] * 100
    daily["MA7"] = daily["SI_PCT"].rolling(7, min_periods=3).mean()
    daily["DELTA"] = daily["SI_PCT"].diff()

    print(f"✅ Stampa: {len(df):,} articoli, {len(daily)} giorni")
    return daily, df


print("Caricamento e costruzione serie giornaliere...")
daily_ref, df_ref_full = load_youtube(PATH_REFERENDUM, "Referendum")
daily_gov, df_gov_full = load_youtube(PATH_GOVERNO,    "Governo")
daily_sta, df_sta_full = load_stampa(PATH_STAMPA)

print("\n📐 Periodi disponibili:")
print(f"  Referendum: {daily_ref.index.min():%d/%m/%Y} → {daily_ref.index.max():%d/%m/%Y}")
print(f"  Governo:    {daily_gov.index.min():%d/%m/%Y} → {daily_gov.index.max():%d/%m/%Y}")
print(f"  Stampa:     {daily_sta.index.min():%d/%m/%Y} → {daily_sta.index.max():%d/%m/%Y}")

## §2 — Plot tre serie sovrapposte (TODO §1.5: "grafico tre linee")

Sovrappongo le tre serie sullo stesso asse temporale. Tre linee con colori distinti:
- 🔵 **Referendum YouTube** (blu) — linea continua spessa
- 🟣 **Governo YouTube** (viola) — linea continua puntinata
- 🟠 **Stampa Camera** (arancione) — linea tratteggiata

Linee di riferimento:
- y = 50% (parità)
- y = 46,1% (risultato reale Sì del referendum)
- x = 22 marzo 2026 (giorno del voto)

Le serie sono lisciate con MA7 *solo per la visualizzazione*. Le serie grezze restano la base per i test econometrici (vedi §3.5.3 della tesi).

In [ ]:
import plotly.graph_objects as go
import os

WORKING_DIR = "/kaggle/working/"

# Periodo comune ai tre dataset
data_start = max(d.index.min() for d in [daily_ref, daily_gov, daily_sta])
data_end   = min(d.index.max() for d in [daily_ref, daily_gov, daily_sta])
print(f"📅 Periodo comune: {data_start:%d/%m/%Y} → {data_end:%d/%m/%Y}")

ref_f = daily_ref[(daily_ref.index >= data_start) & (daily_ref.index <= data_end)]
gov_f = daily_gov[(daily_gov.index >= data_start) & (daily_gov.index <= data_end)]
sta_f = daily_sta[(daily_sta.index >= data_start) & (daily_sta.index <= data_end)]

colori = {
    "Referendum": "#3b82f6",
    "Governo":    "#8b5cf6",
    "Stampa":     "#f59e0b",
}

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ref_f.index, y=ref_f["MA7"],
    name="Referendum YouTube — % Sì",
    line=dict(color=colori["Referendum"], width=3),
    mode="lines",
    hovertemplate="%{x|%d/%m/%Y}<br>Referendum Sì: %{y:.1f}%<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=gov_f.index, y=gov_f["MA7"],
    name="Governo YouTube — % pro-Meloni",
    line=dict(color=colori["Governo"], width=2.5, dash="dot"),
    mode="lines",
    hovertemplate="%{x|%d/%m/%Y}<br>Governo POS: %{y:.1f}%<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=sta_f.index, y=sta_f["MA7"],
    name="Stampa — % articoli pro-riforma",
    line=dict(color=colori["Stampa"], width=2, dash="dash"),
    mode="lines",
    hovertemplate="%{x|%d/%m/%Y}<br>Stampa POS: %{y:.1f}%<extra></extra>",
))

# Linee di riferimento
fig.add_hline(y=50, line_dash="dot", line_color="rgba(0,0,0,0.2)", line_width=1)
fig.add_hline(y=SI_REAL, line_dash="dash", line_color=colori["Referendum"],
              line_width=1.5, annotation_text=f"Voto reale Sì {SI_REAL}%",
              annotation_position="bottom right",
              annotation_font=dict(size=10, color=colori["Referendum"]))
fig.add_vline(x=DATA_VOTO, line_dash="solid", line_color="rgba(0,0,0,0.5)", line_width=2)
fig.add_annotation(x=DATA_VOTO, y=70, text="🗳️ Referendum",
                   showarrow=False, font=dict(size=11))

fig.update_layout(
    title="Confronto stance: Referendum vs Governo vs Stampa<br>"
          "<sub>Media mobile 7gg | YouTube deduplicato | Normalizzato POS vs NEG</sub>",
    xaxis_title="Data", yaxis_title="% pro-Sì / pro-governo / pro-riforma",
    yaxis=dict(range=[20, 80], dtick=10, showgrid=True, gridcolor="rgba(0,0,0,0.08)"),
    xaxis=dict(showgrid=True, gridcolor="rgba(0,0,0,0.05)", tickformat="%d/%m/%y"),
    plot_bgcolor="white", hovermode="x unified", height=600,
    legend=dict(orientation="h", yanchor="bottom", y=1.02,
                xanchor="center", x=0.5),
)
fig.write_html(os.path.join(WORKING_DIR, "tre_linee_sovrapposte.html"))
fig.show(renderer="iframe_connected")
print("💾 Grafico salvato: tre_linee_sovrapposte.html")

## §3 — Plot solo serie referendum (TODO §1.5: "grafico solo stance")

Stesso periodo della §2, ma una sola linea (referendum YouTube) per leggere meglio l'andamento e i picchi senza confronti laterali. Mostra anche la serie grezza in trasparenza per documentare la rumorosità sotto la MA7.

In [ ]:
fig2 = go.Figure()

# Serie grezza in background
fig2.add_trace(go.Scatter(
    x=ref_f.index, y=ref_f["SI_PCT"],
    name="Stance grezza (giornaliera)",
    line=dict(color=colori["Referendum"], width=1),
    opacity=0.35, mode="lines",
    hovertemplate="%{x|%d/%m/%Y}<br>Grezza Sì: %{y:.1f}%<extra></extra>",
))
# MA7 in primo piano
fig2.add_trace(go.Scatter(
    x=ref_f.index, y=ref_f["MA7"],
    name="MA7 (lisciata)",
    line=dict(color=colori["Referendum"], width=3),
    mode="lines",
    hovertemplate="%{x|%d/%m/%Y}<br>MA7 Sì: %{y:.1f}%<extra></extra>",
))

fig2.add_hline(y=50, line_dash="dot", line_color="rgba(0,0,0,0.2)", line_width=1)
fig2.add_hline(y=SI_REAL, line_dash="dash", line_color="#10b981",
               line_width=2, annotation_text=f"Risultato reale: Sì {SI_REAL}%",
               annotation_position="bottom right")
fig2.add_vline(x=DATA_VOTO, line_dash="solid", line_color="rgba(0,0,0,0.5)", line_width=2)
fig2.add_annotation(x=DATA_VOTO, y=ref_f["MA7"].max() + 2, text="🗳️ Voto",
                    showarrow=False, font=dict(size=11))

fig2.update_layout(
    title="Stance pro-Sì su YouTube — referendum costituzionale 2026<br>"
          "<sub>Aggregato deduplicato per autore-giorno | Serie grezza + MA7</sub>",
    xaxis_title="Data", yaxis_title="% pro-Sì",
    yaxis=dict(range=[20, 70], dtick=5, showgrid=True, gridcolor="rgba(0,0,0,0.08)"),
    xaxis=dict(showgrid=True, gridcolor="rgba(0,0,0,0.05)", tickformat="%d/%m/%y"),
    plot_bgcolor="white", hovermode="x unified", height=520,
    legend=dict(orientation="h", yanchor="bottom", y=1.02,
                xanchor="center", x=0.5),
)
fig2.write_html(os.path.join(WORKING_DIR, "solo_stance_referendum.html"))
fig2.show(renderer="iframe_connected")
print("💾 Grafico salvato: solo_stance_referendum.html")

## §4 — Identificazione dei turning points: metodo e formule

**Perché non "smoothing + soglia sulla derivata".** La MA7 *appiattisce i
picchi di un solo giorno* (un video virale che muove la stance per 24–48 h
diventa `Δ/7` spalmato su una settimana → invisibile) ed è *trailing* (ritarda
il livello di ~3 gg); la serie grezza, all'opposto, su giorni con pochi
commenti è dominata dal rumore di campionamento (a `n=5` l'errore sulla % è
≈ ±22 pp). Identificare i punti di svolta è un problema di **changepoint
detection**, che ha metodi propri. **Il numero di turning point non è imposto:
è un output del criterio statistico.**

### Notazione

- Giorno `t = 1, …, N`. `POSₜ`, `NEGₜ` = commenti pro-Sì / pro-No dopo
  deduplicazione; volume `nₜ = POSₜ + NEGₜ ≥ 5`.
- Proporzione grezza pro-Sì: `p̂ₜ = POSₜ / nₜ`.

### 1. Stabilizzazione della varianza (arcoseno-radice)

`p̂ₜ` è una proporzione: la varianza di campionamento dipende da `nₜ`
(eteroschedasticità). Trasformata di Anscombe (1948) / Freeman–Tukey (1950):

```
zₜ = 2 · arcsin( √p̂ₜ )          ⇒          Var(zₜ) ≈ 1 / nₜ
```

Pesare ogni giorno per `wₜ = nₜ` rende il rumore in spazio `z`
(asintoticamente) omoschedastico — è un'**inverse-variance weighting**.

### 2. Segmentazione a media costante (optimal partitioning)

Una segmentazione in `K+1` tratti è data dai break
`0 = τ₀ < τ₁ < … < τ_K < τ_{K+1} = N`; il tratto `k` è `(τ_k , τ_{k+1}]`.
Media e costo (SSE) pesati del tratto:

```
z̄_k   = ( Σ_{t∈k} wₜ zₜ ) / ( Σ_{t∈k} wₜ )

C(a,b) = Σ_{t=a+1}^{b} wₜ ( zₜ − z̄ )²
       = Σ wₜ zₜ²  −  ( Σ wₜ zₜ )² / ( Σ wₜ )      (prefix-sum → O(1))
```

### 3. Numero di break scelto dai dati (penalità BIC)

Si sceglie la segmentazione che minimizza il criterio penalizzato:

```
{τ_k}* = argmin   Σ_{k=0}^{K} C(τ_k , τ_{k+1})   +   β · K
```

con penalità **BIC** `β = c · ln N` (`c = 1` default). Sulla serie a varianza
stabilizzata e pesata, `C` è (asintoticamente) una *devianza* `~ χ²`: ogni
tratto in più aggiunge un solo parametro (la sua media) → l'incremento BIC
canonico è `ln N`. Vincolo di lunghezza minima `τ_{k+1} − τ_k ≥ m`
(`m = 7` osservazioni) ⇒ **persistenza** del cambiamento (sostituisce il
cool-down). `K` non è prefissato.

Soluzione **esatta** per programmazione dinamica (optimal partitioning;
Jackson et al. 2005; PELT, Killick et al. 2012):

```
F(0) = −β
F(j) = min_{ 0 ≤ i ≤ j−m }  [ F(i) + C(i, j) + β ]
```

`F(N)` è l'ottimo; il back-tracking degli `argmin` restituisce i break
`{τ_k}`.

### 4. Lettura dei risultati

Ogni break `τ_k` è un turning point. Livello di stance del tratto, ri-trasforma-
to in % pro-Sì: `p̄_k = sin²( z̄_k / 2 ) · 100`. Direzione del TP:
`segno( p̄_k − p̄_{k−1} )`. La cella stampa anche una **tabella di sensibilità**
del numero di break a `β ∈ {½, 1, 2}·ln N`: se `K` è stabile attorno al valore
BIC, la scelta non è cherry-picked. La MA7 resta confinata ai grafici §2–§3
(sola leggibilità), mai nella statistica.

In [ ]:
def segment_series_weighted(y, w, min_size=7, pen_mult=1.0):
    """Segmentazione a media costante con minimi quadrati PESATI per volume.

    Optimal partitioning esatto (Jackson et al. 2005; Killick et al. 2012),
    nello spirito del test di break multipli di Bai & Perron (2003). Ogni
    giorno t pesa quanto il suo volume w_t (n. commenti): i giorni a basso
    campione contano poco -> robusto all'eteroschedasticita' della % giornaliera.

    Il numero di break minimizza  Σ SSE_pesata  +  pen·(n_break)  con penalita'
    BIC  pen = pen_mult · ln(N). Perche' la SSE pesata sia una *devianza* (cosi'
    che la penalita' BIC sia dimensionalmente corretta) la serie va passata gia'
    a varianza stabilizzata: per una proporzione si usa la trasformata
    arcoseno-radice  z = 2·arcsin(√p)  con  Var(z) ≈ 1/n  → peso w = n
    (Anscombe 1948; Freeman & Tukey 1950). Vincolo: ogni segmento ha almeno
    `min_size` osservazioni (persistenza del cambiamento).

    Returns: lista ordinata di indici di break (inizio del nuovo segmento).
    """
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)
    N = len(y)
    if N < 2 * min_size:
        return []

    # Prefissi per costo pesato O(1):  cost(i,j) = Σwy² − (Σwy)²/Σw
    cw   = np.concatenate([[0.0], np.cumsum(w)])
    cwy  = np.concatenate([[0.0], np.cumsum(w * y)])
    cwyy = np.concatenate([[0.0], np.cumsum(w * y * y)])

    def seg_cost(i, j):                       # segmento [i, j)
        W = cw[j] - cw[i]
        if W <= 0:
            return 0.0
        A = cwy[j] - cwy[i]
        B = cwyy[j] - cwyy[i]
        return B - (A * A) / W

    # La SSE pesata su serie a varianza stabilizzata è già una devianza:
    # penalità BIC dimensionale  pen = pen_mult · ln(N)  per ogni break
    pen = pen_mult * np.log(N)

    # Optimal partitioning esatto, O(N²) (N≈150 → istantaneo)
    F = np.full(N + 1, np.inf)
    F[0] = -pen
    last = np.zeros(N + 1, dtype=int)
    for j in range(min_size, N + 1):
        for i in range(0, j - min_size + 1):
            if not np.isfinite(F[i]):
                continue
            c = F[i] + seg_cost(i, j) + pen
            if c < F[j]:
                F[j] = c
                last[j] = i

    # Backtrack: ricostruisco gli indici di break
    bps, k = [], N
    while k > 0:
        i = last[k]
        if i > 0:
            bps.append(i)
        k = i
    return sorted(bps)


# --- Serie GREZZA giornaliera (NO MA7: appiattisce i picchi di 1 giorno) ---
_si  = ref_f["SI_PCT"].to_numpy(dtype=float)
_tot = ref_f["TOT"].to_numpy(dtype=float)
_mask = np.isfinite(_si) & np.isfinite(_tot) & (_tot > 0)
idx_cp = ref_f.index[_mask]
w_cp   = _tot[_mask]                                  # peso = volume giornaliero

# Trasformata arcoseno-radice: Var(z) ≈ 1/n → la SSE pesata diventa una devianza
p_cp = np.clip(_si[_mask] / 100.0, 1e-6, 1 - 1e-6)
z_cp = 2.0 * np.arcsin(np.sqrt(p_cp))

MIN_SEG = 7
bps = segment_series_weighted(z_cp, w_cp, min_size=MIN_SEG, pen_mult=1.0)

# Livello del segmento = media pesata in spazio z, ri-trasformata in % pro-Sì
bounds = [0] + bps + [len(z_cp)]
seg_lvl = []
for a, b in zip(bounds[:-1], bounds[1:]):
    ww = w_cp[a:b]
    m_z = float(np.sum(ww * z_cp[a:b]) / np.sum(ww))
    seg_lvl.append(np.sin(m_z / 2.0) ** 2 * 100.0)

tp_auto = []
for s, bp in enumerate(bps, start=1):
    delta = seg_lvl[s] - seg_lvl[s - 1]
    tipo = "📈 Sì sale" if delta > 0 else "📉 No sale"
    tp_auto.append((pd.Timestamp(idx_cp[bp]), round(seg_lvl[s], 1), tipo, abs(delta)))

print(f"🎯 Turning points (break a media costante, penalità BIC): {len(tp_auto)}")
print(f"   Serie grezza giornaliera pesata per volume | min {MIN_SEG} oss./segmento")
print("=" * 82)
print(f"{'Data':<12} {'Livello dopo':>14} {'Direzione':<14} {'Δ livello':>12}")
print("-" * 82)
for d, v, tipo, amp in tp_auto:
    print(f"{d:%d/%m/%Y}  {v:12.1f}%  {tipo:<14} {amp:9.1f}pp")

# Robustezza: il numero di break NON è tarato a mano
print()
print("🔎 Sensibilità del numero di break alla penalità BIC:")
for pm in (0.5, 1.0, 2.0):
    n_bp = len(segment_series_weighted(z_cp, w_cp, min_size=MIN_SEG, pen_mult=pm))
    flag = "  ← usato (BIC)" if abs(pm - 1.0) < 1e-9 else ""
    print(f"   pen ×{pm:>4}:  {n_bp} break{flag}")

## §5 — Verifica falsificabile di H2

**Ipotesi H2** (Cap 1): almeno il **70%** dei turning point *statisticamente
rilevanti* corrisponde a un evento mediatico documentabile.

Con il metodo §4 il numero di break **esce dai dati** (criterio BIC), non è
fissato a 7: questo rende il test di H2 *non circolare*. Valuto due direzioni
distinte, con tolleranza **±3 giorni** — qui giustificata come *lag fisiologico*
fra evento mediatico e reazione nei commenti (non più come artefatto del
ritardo della MA7, che con la serie grezza non esiste):

- **(A) Precisione → è il test vero di H2.** Dei `N` break individuati
  dall'algoritmo, quale frazione cade entro ±3 gg da un evento del repertorio
  documentato? H2 è **confermata se ≥ 70%**.
- **(B) Recall → diagnostica complementare.** Degli eventi del repertorio
  documentato (Cap 5, Tabella 5.1), quanti vengono ritrovati dall'algoritmo?

Se la precisione (A) scende sotto il 70%, **H2 è falsificata**: è questo che
rende l'ipotesi scientificamente seria.

In [ ]:
# Repertorio di eventi documentati (Cap 5, Tabella 5.1)
TP_TESI = [
    ("2025-10-30", 42.6, "📉 No sale",  "Approvazione al Senato"),
    ("2025-11-19", 51.5, "📈 Sì sale",  "Controffensiva mediatica centrodestra"),
    ("2026-01-13", 52.1, "📈 Sì sale",  "Endorsement Marco Rizzo"),
    ("2026-01-23", 49.4, "📉 No sale",  "Effetto Barbero — primo video"),
    ("2026-02-10", 42.7, "📉 Picco No", "Barbero #2 + Travaglio"),
    ("2026-02-27", 50.5, "📈 Sì sale",  "Rivelazioni Di Pietro"),
    ("2026-03-15", 49.4, "📉 No def.",  "Travaglio + Gratteri settimana finale"),
]
EVENTI = [(pd.to_datetime(d), ev) for d, _, _, ev in TP_TESI]
TOL_GG = 3

# ---- (A) PRECISIONE: dei N break data-driven, quanti presso un evento? (test H2) ----
print("=" * 95)
print("(A) PRECISIONE — ogni break individuato vs evento documentato più vicino")
print("=" * 95)
print(f"{'Data break':<12} {'Livello':>8} {'Direzione':<13} {'Δgg':>5}  Evento associato")
print("-" * 95)
n_map = 0
for d, v, tipo, amp in tp_auto:
    vicini = sorted(((abs((d - de).days), ev) for de, ev in EVENTI),
                    key=lambda t: t[0])
    dgg, ev = vicini[0] if vicini else (None, None)
    mapped = dgg is not None and dgg <= TOL_GG
    n_map += int(mapped)
    flag = "✅" if mapped else "❌"
    ev_str = ev if mapped else "— (nessun evento entro ±%dgg)" % TOL_GG
    print(f"{d:%d/%m/%Y}  {v:7.1f}%  {tipo:<13} {flag}{dgg:>3}  {ev_str}")

precisione = n_map / len(tp_auto) * 100 if tp_auto else 0.0
H2_ok = precisione >= 70.0
print("-" * 95)
print(f"📊 Precisione: {n_map}/{len(tp_auto)} = {precisione:.0f}%  →  "
      f"H2 {'CONFERMATA ✅' if H2_ok else 'FALSIFICATA ❌'} (soglia 70%)")

# ---- (B) RECALL: eventi del repertorio Cap 5 ritrovati dall'algoritmo ----
print()
print("=" * 95)
print("(B) RECALL — eventi noti del Cap 5 ritrovati dall'algoritmo")
print("=" * 95)
print(f"{'Data evento':<12} {'Atteso':>7}  {'Break più vicino':>18} {'Δgg':>5}  Evento")
print("-" * 95)
n_rec = 0
for (data_str, atteso_pct, _dir, evento) in TP_TESI:
    de = pd.to_datetime(data_str)
    near = sorted(tp_auto, key=lambda t: abs((t[0] - de).days))
    closest = near[0] if near else None
    dgg = (closest[0] - de).days if closest else None
    found = closest is not None and abs(dgg) <= TOL_GG
    n_rec += int(found)
    flag = "✅" if found else "❌"
    cstr = f"{closest[0]:%d/%m/%Y}" if closest else "—"
    dstr = f"{dgg:+d}" if dgg is not None else "—"
    print(f"{de:%d/%m/%Y}  {atteso_pct:>6.1f}%  {flag} {cstr:<16} {dstr:>5}  {evento}")

recall = n_rec / len(TP_TESI) * 100
print("-" * 95)
print(f"📊 Recall: {n_rec}/{len(TP_TESI)} = {recall:.0f}%  "
      f"(diagnostica: copertura del repertorio narrativo del Cap 5)")

## §6 — Top video YouTube nelle finestre dei turning points

Per ciascun evento del repertorio documentato (Cap 5, Tabella 5.1): estraggo i 5 video
più commentati nella finestra **±2 giorni** intorno alla data. È la lettura sostantiva
di Cap 5: cosa stava guardando il pubblico mentre la stance si stava muovendo?

In aggiunta, segnalo i video **pubblicati** nella finestra (non solo riattivati): sono
gli eventi mediatici che hanno *originato* il movimento, distinti da contenuti
preesistenti che hanno solo accumulato commenti.

In [ ]:
# Riprendo df_ref_full (con tutte le colonne) per le finestre
df_ref_full["Data_Video_P"] = pd.to_datetime(
    df_ref_full.get("Data_Video"), errors="coerce", utc=True
).dt.tz_localize(None)

print("=" * 95)
print("🎥 TOP VIDEO YOUTUBE — finestra ±2 giorni intorno a ciascun turning point")
print("=" * 95)

for data_str, atteso_pct, direzione, evento in TP_TESI:
    data_evento = pd.to_datetime(data_str)
    win_start = data_evento - pd.Timedelta(days=2)
    win_end   = data_evento + pd.Timedelta(days=2)

    df_win = df_ref_full[(df_ref_full["Data"] >= win_start) &
                         (df_ref_full["Data"] <= win_end)]
    if df_win.empty:
        print(f"\n📅 {data_evento:%d/%m/%Y} — {evento}: nessun commento")
        continue

    print(f"\n📅 {data_evento:%d/%m/%Y} — {direzione}: {evento}")
    print(f"   Commenti totali nella finestra: {len(df_win):,}")

    # Top 5 video per volume
    top_video = (df_win.groupby("Video_ID")
                 .agg(Titolo=("Titolo_Video", "first"),
                      Canale=("Canale", "first"),
                      N=("STANCE", "size"),
                      POS=("STANCE", lambda x: (x == "POS").sum()),
                      NEG=("STANCE", lambda x: (x == "NEG").sum()))
                 .nlargest(5, "N"))
    print("   🔥 Top video più commentati:")
    for _, v in top_video.iterrows():
        si = v["POS"] / max(1, v["POS"] + v["NEG"]) * 100
        print(f"      [{str(v['Canale'])[:22]:<22}] {str(v['Titolo'])[:60]}")
        print(f"         💬 {v['N']:>5} (Sì {si:.0f}%)")

    # Video pubblicati nella finestra (eventi originanti)
    nuovi = df_win[(df_win["Data_Video_P"] >= win_start) &
                   (df_win["Data_Video_P"] <= win_end)]
    if not nuovi.empty:
        nuovi_unique = nuovi["Video_ID"].drop_duplicates()
        print(f"   🆕 Video PUBBLICATI in questi giorni: {len(nuovi_unique)}")
        top_nuovi = (nuovi.groupby("Video_ID")
                     .agg(Titolo=("Titolo_Video", "first"),
                          Canale=("Canale", "first"),
                          N=("STANCE", "size"))
                     .nlargest(3, "N"))
        for _, v in top_nuovi.iterrows():
            print(f"      ★ [{str(v['Canale'])[:22]:<22}] {str(v['Titolo'])[:60]} ({v['N']} com.)")

## §7 — Articoli stampa nelle finestre dei turning points

Lo stesso esercizio della §6, ma sui titoli della stampa. Finestra **±7 giorni** anziché
±2: la motivazione è il lag empirico atteso fra stampa e referendum YouTube
(stimato in 3-7 giorni dai test di Granger del Cap 6). Mostro i titoli più rappresentativi
del polo prevalente nella finestra, con le testate più attive.

In [ ]:
titolo_col  = next((c for c in df_sta_full.columns if "titolo" in c.lower()), None)
testata_col = next((c for c in df_sta_full.columns if "testata" in c.lower()), None)

print("=" * 95)
print("📰 ARTICOLI DI STAMPA — finestra ±7 giorni intorno a ciascun turning point")
print("=" * 95)

for data_str, atteso_pct, direzione, evento in TP_TESI:
    data_evento = pd.to_datetime(data_str)
    win_start = data_evento - pd.Timedelta(days=7)
    win_end   = data_evento + pd.Timedelta(days=7)

    df_win = df_sta_full[(df_sta_full["Data"] >= win_start) &
                         (df_sta_full["Data"] <= win_end)]
    if df_win.empty:
        print(f"\n📅 {data_evento:%d/%m/%Y} — {evento}: nessun articolo")
        continue

    pos = (df_win["STANCE"] == "POS").sum()
    neg = (df_win["STANCE"] == "NEG").sum()
    si_pct = pos / max(1, pos + neg) * 100

    print(f"\n📅 {data_evento:%d/%m/%Y} — {direzione}: {evento}")
    print(f"   Stampa nella finestra ±7gg: {len(df_win):,} articoli (Sì {si_pct:.0f}%)")

    # Top 5 testate
    top_t = df_win[testata_col].value_counts().head(5)
    if len(top_t) > 0:
        print("   📰 Testate più attive:")
        for testata, n in top_t.items():
            sub = df_win[df_win[testata_col] == testata]
            p = (sub["STANCE"] == "POS").sum()
            ng = (sub["STANCE"] == "NEG").sum()
            print(f"      {str(testata)[:25]:<25}  {n:>4} art.  (Sì {p}, No {ng})")

    # 3 titoli del polo prevalente
    polo = "POS" if si_pct >= 50 else "NEG"
    polo_label = "pro-riforma" if polo == "POS" else "contro-riforma"
    df_polo = df_win[df_win["STANCE"] == polo].head(3)
    if not df_polo.empty:
        marker = "🟢" if polo == "POS" else "🔴"
        print(f"   {marker} Esempi titoli {polo_label}:")
        for _, r in df_polo.iterrows():
            print(f"      [{str(r[testata_col])[:18]:<18}] {str(r[titolo_col])[:70]}")

## §8 — Parole chiave: commenti vs titoli stampa per ciascun TP

Estrazione delle parole più frequenti (filtrate da stopword italiane e termini di dominio
generici) nei commenti YouTube e nei titoli stampa della finestra di ciascun turning point.

L'utilità è leggere la **divergenza tematica**: spesso il pubblico online discute aspetti
diversi (concretezza, sfiducia, esempi personali) rispetto alla stampa (terminologia
istituzionale, dichiarazioni ufficiali). Il confronto diventa il materiale narrativo per
la sezione *5.4 Pattern emergenti* della tesi.

In [ ]:
import re
from collections import Counter

stop_it = set([
    "il", "lo", "la", "i", "gli", "le", "un", "uno", "una", "di", "a", "da",
    "in", "con", "su", "per", "tra", "fra", "che", "e", "è", "non", "ha",
    "ho", "sono", "ma", "se", "come", "più", "al", "del", "ci", "si", "mi",
    "ti", "vi", "o", "questo", "quello", "alla", "della", "dei", "degli",
    "delle", "alle", "ai", "nel", "nella", "nei", "nelle", "hanno", "fatto",
    "fare", "chi", "cui", "quale", "quali", "tutti", "tutto", "solo", "anche",
    "essere", "stato", "stata", "stati", "state", "quindi", "perché",
    "referendum", "giustizia", "voto", "votare", "sì", "no", "riforma",
    "contro", "favore", "italia", "italiani", "governo", "meloni",
    "magistratura", "magistrato", "magistrati", "carriere", "separazione",
    "essere", "avere", "andare", "venire", "potere", "dovere", "fatto",
    "detto", "video", "canale",
])


def parole_top(testi, top=10):
    cnt = Counter()
    for t in testi.dropna():
        t = re.sub(r"[^a-zA-Zàèéìòùç\s]", " ", str(t).lower())
        for w in t.split():
            if w not in stop_it and len(w) > 3:
                cnt[w] += 1
    return cnt.most_common(top)


testo_yt = "Testo" if "Testo" in df_ref_full.columns else "Commento"

print("=" * 95)
print("🔑 PAROLE CHIAVE — commenti YouTube vs titoli stampa per turning point")
print("=" * 95)

for data_str, atteso_pct, direzione, evento in TP_TESI:
    data_evento = pd.to_datetime(data_str)
    win_yt_start = data_evento - pd.Timedelta(days=2)
    win_yt_end   = data_evento + pd.Timedelta(days=2)
    win_st_start = data_evento - pd.Timedelta(days=7)
    win_st_end   = data_evento + pd.Timedelta(days=7)

    yt_win = df_ref_full[(df_ref_full["Data"] >= win_yt_start) &
                         (df_ref_full["Data"] <= win_yt_end)]
    st_win = df_sta_full[(df_sta_full["Data"] >= win_st_start) &
                         (df_sta_full["Data"] <= win_st_end)]

    parole_yt = parole_top(yt_win[testo_yt], top=8)
    parole_st = parole_top(st_win[titolo_col], top=8) if titolo_col else []

    print(f"\n📅 {data_evento:%d/%m/%Y} — {evento}")
    if parole_yt:
        print(f"   🔴 YouTube  ({len(yt_win):>5} com.):  "
              + " | ".join(f"{w}({n})" for w, n in parole_yt))
    if parole_st:
        print(f"   📰 Stampa   ({len(st_win):>5} art.):  "
              + " | ".join(f"{w}({n})" for w, n in parole_st))

## §9 — Top picchi e inversioni della serie stampa

Anticipo (preview) per il Capitolo 6: identifico i giorni più "polarizzati" della stampa
(massimi e minimi della MA7 pro-riforma) e le **inversioni di tendenza** (giornate in cui
la maggioranza pro-Sì supera la maggioranza pro-No, o viceversa).

Questa diagnostica rende leggibile il segnale che entrerà nel test di Granger come variabile
*causa*: prima di chiedersi se la stampa Granger-causi YouTube, vale la pena di vedere
quando e dove la stampa stessa si è mossa.

In [ ]:
sta_f = sta_f.copy()
sta_f["VINCITORE"] = np.where(sta_f["MA7"] > 50, "POS", "NEG")
sta_f["INVERSIONE"] = sta_f["VINCITORE"] != sta_f["VINCITORE"].shift(1)

picchi_max = sta_f.nlargest(10, "MA7")
picchi_min = sta_f.nsmallest(10, "MA7")
inversioni = sta_f[sta_f["INVERSIONE"]].iloc[1:]   # salto il primo (NaN sull'iniziale)

print("=" * 75)
print("🔝 TOP 10 GIORNI — STAMPA PIÙ PRO-RIFORMA")
print("=" * 75)
for d, row in picchi_max.iterrows():
    print(f"  {d:%d/%m/%Y}  Sì {row['MA7']:.1f}%  ({int(row['TOT'])} articoli)")

print("\n" + "=" * 75)
print("🔻 TOP 10 GIORNI — STAMPA MENO PRO-RIFORMA")
print("=" * 75)
for d, row in picchi_min.iterrows():
    print(f"  {d:%d/%m/%Y}  Sì {row['MA7']:.1f}%  ({int(row['TOT'])} articoli)")

print("\n" + "=" * 75)
print("🔄 INVERSIONI DI TENDENZA STAMPA (POS↔NEG)")
print("=" * 75)
if len(inversioni) == 0:
    print("  Nessuna inversione: la stampa è rimasta sopra il 50% per tutto il periodo.")
else:
    for d, row in inversioni.iterrows():
        direzione = "POS supera NEG ✅" if row["VINCITORE"] == "POS" \
                    else "NEG supera POS ❌"
        print(f"  {d:%d/%m/%Y}  {direzione}  (MA7: {row['MA7']:.1f}%)")

# Salvataggio dataset master per il notebook 05 (Granger)
master_data = pd.DataFrame({
    "Referendum": ref_f["SI_PCT"],
    "Governo":    gov_f["SI_PCT"],
    "Stampa":     sta_f["SI_PCT"],
})
master_data.index.name = "Data"
master_data.to_csv(os.path.join(WORKING_DIR, "serie_giornaliere_master.csv"),
                   encoding="utf-8-sig")
print(f"\n💾 Salvato serie_giornaliere_master.csv per notebook 05 ({len(master_data)} giorni)")

## Sintesi del notebook

| Risultato | Atteso (Cap 5) | Osservato (riempire dopo run) |
|---|---|---|
| Periodo comune 3 serie | ott 2025 – mar 2026 | _da inserire_ |
| TP automatici (break BIC, data-driven) | N scelto dai dati | _da inserire_ |
| Precisione H2 (break presso evento, ±3gg) | ≥ 70% → H2 ok | _da inserire_ |
| Recall (7 eventi Cap 5 ritrovati, ±3gg) | diagnostica | _da inserire_ |
| Top video Barbero #1 (23/01) | nei top-5 commentati | _da inserire_ |
| Top video Barbero #2 (10/02) | nei top-5 commentati | _da inserire_ |
| Stampa: % giorni con MA7 ≥ 50 | 90-100% | _da inserire_ |
| Inversioni stampa | 0-2 episodi | _da inserire_ |

**File salvati in `/kaggle/working/`:**
- `tre_linee_sovrapposte.html` — grafico TODO §1.5 "tre linee"
- `solo_stance_referendum.html` — grafico TODO §1.5 "grafico solo stance"
- `serie_giornaliere_master.csv` — input per il notebook 05 (Granger)

**Prossimo notebook:** `05_granger_causality.ipynb` (alimenta Cap 6: ADF + Granger su 6 coppie + heatmap).